# System Dependencies

To get started with Unstructured.io, we need a few system-wide dependencies: 

## Poppler (poppler-utils)
Handles PDF processing. It's a library that can extract text, images, and metadata from PDFs. Unstructured uses it to parse PDF documents and convert them into processable text.

## Tesseract (tesseract-ocr) 
Optical Character Recognition (OCR) engine. When you have scanned documents, images with text, or PDFs that are essentially pictures, Tesseract reads the text from these images and converts it to machine-readable text.

## libmagic
File type detection library. It identifies what type of file you're dealing with (PDF, Word doc, image, etc.) by analyzing the file's content, not just the extension. This helps Unstructured choose the right processing method for each document.

In [1]:
# for linux
# !sudo apt-get install -y poppler-utils tesseract-ocr libmagic-dev

# for mac
# !brew install poppler tesseract libmagic

In [2]:
%pip install -q "unstructured[all-docs]"
%pip install -q langchain_chroma
%pip install -q langchain langchain-groq langchain-openrouter langchain-openai
%pip install -q python_dotenv

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import os
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_groq import ChatGroq
from langchain_openrouter import ChatOpenRouter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from openrouter.errors import ResponseValidationError, TooManyRequestsResponseError
from dotenv import load_dotenv

load_dotenv()

# Models (keys come from .env: GROQ_API_KEY, OPENROUTER_API_KEY)
GENERATION_MODEL = "qwen/qwen3.8-27b"  # Groq, answers the question
IMAGE_DESCRIPTION_MODEL = "dots-studio/dots-3-note-preview:free"  # OpenRouter, describes images and tables
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b:free"  # OpenRouter, text embeddings


class NemotronEmbeddings(Embeddings):
    """Nemotron on OpenRouter. It expects input_type=passage for chunks and input_type=query for questions."""

    def _model(self, input_type):
        return OpenAIEmbeddings(
            model=EMBEDDING_MODEL,
            api_key=os.environ["OPENROUTER_API_KEY"],
            base_url="https://openrouter.ai/api/v1",
            check_embedding_ctx_length=False,
            model_kwargs={"extra_body": {"input_type": input_type}},
        )

    def embed_documents(self, texts):
        return self._model("passage").embed_documents(texts)

    def embed_query(self, text):
        return self._model("query").embed_query(text)

/teamspace/studios/this_studio/multimodal-rag-pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements

# Test with your PDF file
file_path = "../attention-is-all-you-need-first6.pdf"  # First 6 pages only, for a quick test run
elements = partition_document(file_path)

📄 Partitioning document: ../attention-is-all-you-need-first6.pdf


No languages specified, defaulting to English.


Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

Loading weights:  45%|████▌     | 166/367 [00:00<00:00, 1641.70it/s]

Loading weights: 100%|██████████| 367/367 [00:00<00:00, 2575.47it/s]

✅ Extracted 112 elements


In [5]:
# elements
# len(elements)


elements

In [6]:
# All types of different atomic elements we see from unstructured
set([str(type(el)) for el in elements])

{"<class 'unstructured.documents.elements.FigureCaption'>",
 "<class 'unstructured.documents.elements.Footer'>",
 "<class 'unstructured.documents.elements.Formula'>",
 "<class 'unstructured.documents.elements.Header'>",
 "<class 'unstructured.documents.elements.Image'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [7]:
elements[36].to_dict()

{'type': 'NarrativeText',
 'element_id': 'c9a6794b5dc35dbe96d655aecbed7872',
 'text': 'Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been firmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation [35, 2, 5]. Numerous efforts have since continued to push the boundaries of recurrent language models and encoder-decoder architectures [38, 24, 15].',
 'metadata': {'detection_class_prob': 0.9414572715759277,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(518.5806884765625),
     np.float64(480.4494799999996)),
    (np.float64(518.5806884765625), np.float64(740.9982300000003)),
    (np.float64(2466.774658203125), np.float64(740.9982300000003)),
    (np.float64(2466.774658203125), np.float64(480.4494799999996))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-10-

In [8]:
# Gather all images
images = [element for element in elements if element.category == 'Image']
print(f"Found {len(images)} images")

images[0].to_dict()

# Use https://codebeautify.org/base64-to-image-converter to view the base64 text

Found 3 images


{'type': 'Image',
 'element_id': '018566ecccb6923695913bad17022c06',
 'text': 'Output Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention a, Add & Norm Nx Add & Norm Feed Forward Nx | —-Casda Nom] Add & Norm VWEeea Multi-Head Multi-Head Attention Attention Se a, SE a, Positional CY Encoding or, Positional @ g Encoding Input Embedding Inputs Outputs (shifted right) Output Embedding',
 'metadata': {'coordinates': {'points': ((np.float64(955.4951388888888),
     np.float64(350.00972222222197)),
    (np.float64(955.4951388888888), np.float64(1917.309722222222)),
    (np.float64(2019.4951388888885), np.float64(1917.309722222222)),
    (np.float64(2019.4951388888885), np.float64(350.00972222222197))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-10-12T16:06:17',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 3,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory':

In [9]:
# Gather all table
tables = [element for element in elements if element.category == 'Table']
print(f"Found {len(tables)} tables")

tables[0].to_dict()

# Use https://jsfiddle.net/ to view the table html 


Found 1 tables


{'type': 'Table',
 'element_id': '53df85a3e2288d738e284d263d7e60e1',
 'text': 'Layer Type Complexity per Layer Sequential Maximum Path Length Operations Self-Attention O(n2 · d) O(1) O(1) Recurrent O(n · d2) O(n) O(n) Convolutional O(k · n · d2) O(1) O(logk(n)) Self-Attention (restricted) O(r · n · d) O(1) O(n/r)',
 'metadata': {'detection_class_prob': 0.9282334446907043,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(561.3707885742188),
     np.float64(550.45751953125)),
    (np.float64(561.3707885742188), np.float64(908.8243408203125)),
    (np.float64(2395.140380859375), np.float64(908.8243408203125)),
    (np.float64(2395.140380859375), np.float64(550.45751953125))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-10-12T16:06:17',
  'text_as_html': '<table><thead><tr><th>Layer Type</th><th>Complexity per Layer</th><th>Sequential Operations</th><th>Maximum Path Length</th></tr></thead><tbody><tr><td>Self-Att

In [10]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks
chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 15 chunks


In [11]:
# View all chunks
# chunks

# All unique types
set([str(type(chunk)) for chunk in chunks])

{"<class 'unstructured.documents.elements.CompositeElement'>",
 "<class 'unstructured.documents.elements.Table'>"}

In [12]:
# View a single chunk
# chunks[2].to_dict()

# View original elements
chunks[11].metadata.orig_elements[-1].to_dict()
# Note: 4th chunk has the first image + 11th chunk has the first table in the sample PDF

{'type': 'NarrativeText',
 'element_id': '685a6733f6055d323e14d53981f144b6',
 'text': 'Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations for different layer types. n is the sequence length, d is the representation dimension, k is the kernel size of convolutions and r the size of the neighborhood in restricted self-attention.',
 'metadata': {'detection_class_prob': 0.9406974911689758,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(520.6363525390625),
     np.float64(344.8682299999998)),
    (np.float64(520.6363525390625), np.float64(499.3572577777778)),
    (np.float64(2465.873779296875), np.float64(499.3572577777778)),
    (np.float64(2465.873779296875), np.float64(344.8682299999998))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-10-12T16:06:17',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 6,
  'file_directory': '..',
  'filename': 

In [13]:
def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            
            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            
            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    
    content_data['types'] = list(set(content_data['types']))
    return content_data

def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content"""
    
    try:
        # Initialize LLM (needs vision model for images)
        # The free OpenRouter route sometimes answers "provider unavailable" or 429 rate-limited, so retry those errors
        # max_tokens stops a runaway reply; the embedding model accepts at most 4096 tokens per chunk
        llm = ChatOpenRouter(model=IMAGE_DESCRIPTION_MODEL, temperature=0, max_tokens=3000).with_retry(
            retry_if_exception_type=(ResponseValidationError, TooManyRequestsResponseError), stop_after_attempt=8
        )
        
        # Build the text prompt
        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}

        """
        
        # Add tables if present
        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"

        # Task instructions apply to every chunk, with or without tables
        prompt_text += """
        YOUR TASK:
        Generate a comprehensive, searchable description that covers:

        1. Key facts, numbers, and data points from text and tables
        2. Main topics and concepts discussed  
        3. Questions this content could answer
        4. Visual content analysis (charts, diagrams, patterns in images)
        5. Alternative search terms users might use

        Make it detailed and searchable - prioritize findability over brevity.

        SEARCHABLE DESCRIPTION:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add images to the message
        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        # Fallback to simple summary
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")
    
    langchain_documents = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")
        
        # Analyze chunk content
        content_data = separate_content_types(chunk)
        
        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        
        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'], 
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        
        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        
        langchain_documents.append(doc)
    
    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents


# Process chunks with AI
processed_chunks = summarise_chunks(chunks)

🧠 Processing chunks with AI Summaries...
   Processing chunk 1/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 2/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 3/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 4/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 5/15
     Types found: ['image', 'text']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...


     → AI summary created successfully
     → Enhanced content preview: 

### SEARCHABLE DESCRIPTION

---

### 1. Key Facts, Numbers, and Data Points
*   **Model Type:** The Transformer is a neural sequence transduction model featuring an **encoder-decoder structure** (ba...
   Processing chunk 6/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 7/15
     Types found: ['image', 'text']
     Tables: 0, Images: 2
     → Creating AI summary for mixed content...


     → AI summary created successfully
     → Enhanced content preview: 

### Searchable Description: Scaled Dot-Product and Multi-Head Attention (Section 3.2)

---

#### 1. Key Facts, Numbers, and Core Data Points
*   **Core Definition of Attention:** An attention functi...
   Processing chunk 8/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 9/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 10/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 11/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 12/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 13/15
     Types found: ['text', 'table']
     Tables: 1, Images: 0
     → Creating AI summary for mixed c

     → AI summary created successfully
     → Enhanced content preview: 

SEARCHABLE DESCRIPTION:

This document provides a comparative analysis of computational complexity and operational characteristics for different neural network layer types, specifically focusing on ...
   Processing chunk 14/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 15/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
✅ Processed 15 chunks


In [14]:
processed_chunks

[Document(metadata={'original_content': '{"raw_text": "3\\n\\n2023\\n\\n2\\n\\n0\\n\\n2 g u A 2 ] L C . s c [ 7 v 2 6 7 3 0 . 6 0 7 1 :\\n\\nv\\n\\narX1V\\n\\ni\\n\\nX\\n\\nr\\n\\na\\n\\nProvided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\\n\\nAttention Is All You Need\\n\\nAshish Vaswani\\u2217\\n\\nGoogle Brain\\n\\navaswani@google.com\\n\\nNoam Shazeer\\u2217\\n\\nGoogle Brain noam@google.com\\n\\nNiki Parmar\\u2217\\n\\nGoogle Research nikip@google.com\\n\\nJakob Uszkoreit\\u2217\\n\\nGoogle Research usz@google.com\\n\\nLlion Jones\\u2217\\n\\nGoogle Research\\n\\nllion@google.com\\n\\nAidan N. Gomez\\u2217 \\u2020 University of Toronto aidan@cs.toronto.edu\\n\\n\\u0141ukasz Kaiser\\u2217 Google Brain lukaszkaiser@google.com", "tables_html": [], "images_base64": []}'}, page_content='3\n\n2023\n\n2\n\n0\n\n2 g u A 2 ] L C . s c [ 7 v 2 6 7 3 0 . 6 0 7 1 :\n\nv\n\

In [15]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

✅ Exported 15 chunks to chunks_export.json


In [16]:
def create_vector_store(documents, persist_directory="dbv1_first6/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("🔮 Creating embeddings and storing in ChromaDB...")
        
    embedding_model = NemotronEmbeddings()
    
    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory, 
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")
    
    print(f"✅ Vector store created and saved to {persist_directory}")
    return vectorstore

# Create the vector store
db = create_vector_store(processed_chunks)

🔮 Creating embeddings and storing in ChromaDB...
--- Creating vector store ---


--- Finished creating vector store ---
✅ Vector store created and saved to dbv1_first6/chroma_db


In [17]:
# After your retrieval
query = "What are the two main components of the Transformer architecture? "
retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

# Export to JSON
export_chunks_to_json(chunks, "rag_results.json")

✅ Exported 3 chunks to rag_results.json


[{'chunk_id': 1,
  'enhanced_content': '\n\n### SEARCHABLE DESCRIPTION\n\n---\n\n### 1. Key Facts, Numbers, and Data Points\n*   **Model Type:** The Transformer is a neural sequence transduction model featuring an **encoder-decoder structure** (based on references [5, 2, 35]).\n*   **Input Representation:** Maps an input sequence of symbol representations $(x_1, ..., x_n)$ to continuous representations $z = (z_1, ..., z_n)$ via the encoder.\n*   **Output Generation:** Generates an output sequence $(y_1, ..., y_m)$ one element at a time.\n*   **Mechanism:** Auto-regressive [10], consuming previously generated symbols as additional input when generating the next.\n*   **Core Layers:** Uses stacked **self-attention** and **point-wise, fully connected (Feed Forward) layers** for both the encoder (left half of Figure 1) and decoder (right half of Figure 1).\n*   **Stacking Factor ($Nx$):** Both the encoder and decoder blocks repeat $Nx$ times, representing $N$ stacked layers.\n*   **Key Com

In [18]:
def run_complete_ingestion_pipeline(pdf_path: str):
    """Run the complete RAG ingestion pipeline"""
    print("🚀 Starting RAG Ingestion Pipeline")
    print("=" * 50)
    
    # Step 1: Partition
    elements = partition_document(pdf_path)
    
    # Step 2: Chunk
    chunks = create_chunks_by_title(elements)
    
    # Step 3: AI Summarisation
    summarised_chunks = summarise_chunks(chunks)
    
    # Step 4: Vector Store
    db = create_vector_store(summarised_chunks, persist_directory="dbv2_first6/chroma_db")
    
    print("🎉 Pipeline completed successfully!")
    return db

# Run the complete pipeline

In [19]:
db = run_complete_ingestion_pipeline("../attention-is-all-you-need-first6.pdf")

🚀 Starting RAG Ingestion Pipeline
📄 Partitioning document: ../attention-is-all-you-need-first6.pdf


No languages specified, defaulting to English.


✅ Extracted 112 elements
🔨 Creating smart chunks...
✅ Created 15 chunks
🧠 Processing chunks with AI Summaries...
   Processing chunk 1/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 2/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 3/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 4/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 5/15
     Types found: ['image', 'text']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...


     → AI summary created successfully
     → Enhanced content preview: 

### Searchable Description

---

#### 1. Key Facts, Numbers, and Data Points
*   **Core Architecture Type:** Encoder-decoder structure, typical of state-of-the-art neural sequence transduction model...
   Processing chunk 6/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 7/15
     Types found: ['image', 'text']
     Tables: 0, Images: 2
     → Creating AI summary for mixed content...


     → AI summary created successfully
     → Enhanced content preview: 

### **SEARCHABLE DESCRIPTION**

---

### **1. Key Facts, Numbers, and Data Points**
* **Section Reference:** Section 3.2 ("Attention") of a technical document (likely the "Attention Is All You Need"...
   Processing chunk 8/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 9/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 10/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 11/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 12/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 13/15
     Types found: ['text', 'table']
     Tables: 1, Images: 0
     → Creating AI summary for mixed c

     → AI summary created successfully
     → Enhanced content preview: 

Of course. Here is a comprehensive, searchable description based on the provided content.

***

### **SEARCHABLE DESCRIPTION: Neural Network Layer Complexity Comparison**

**1. Key Facts, Numbers, a...
   Processing chunk 14/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 15/15
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
✅ Processed 15 chunks
🔮 Creating embeddings and storing in ChromaDB...
--- Creating vector store ---


--- Finished creating vector store ---
✅ Vector store created and saved to dbv2_first6/chroma_db
🎉 Pipeline completed successfully!


In [20]:
# Query the vector store
query = "How many attention heads does the Transformer use, and what is the dimension of each head? "

retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

def generate_final_answer(chunks, query):
    """Generate final answer using multimodal content"""
    
    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatGroq(model=GENERATION_MODEL, temperature=0, reasoning_format="parsed")
        
        # Build the text prompt
        prompt_text = f"""Based on the following documents, please answer this question: {query}

CONTENT TO ANALYZE:
"""
        
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"
            
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                
                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                
                # Add tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"
            
            prompt_text += "\n"
        
        prompt_text += """
Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."

ANSWER:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add all images from all chunks
        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("images_base64", [])
                
                for image_base64 in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."

# Usage
final_answer = generate_final_answer(chunks, query)
print(final_answer)

Based on **Document 2**, under the section titled **“3.2.2 Multi-Head Attention”**, the text explicitly states:

> “In this work we employ **h = 8** parallel attention layers, or heads. For each of these we use **dk = dv = dmodel/h = 64**.”

This directly answers the question.

---

### ✅ Final Answer:

The Transformer uses **8 attention heads**.  
The dimension of each head is **64** (for both keys and values, i.e., `dk = dv = 64`).

This is derived from the model dimension `dmodel` being divided by the number of heads `h`:  
`dmodel / h = 64` → which implies `dmodel = 512` (since 512 / 8 = 64), though the question only asks for the number of heads and the per-head dimension.

---

**Note**: While Document 1 describes *how* multi-head attention is used (encoder self-attention, decoder self-attention, encoder-decoder attention), it does not specify the number of heads or their dimensions. Document 3 provides the overall architecture diagram but also lacks specific numerical parameters.